In [0]:
from pyspark.sql import functions as F
from config import load_config

cfg = load_config("config_databricks.yaml")

fct_workflow_events = (
    spark.table(cfg["bronze_workflow_events_table"])
    .select(
        F.trim(F.col("event_id")).alias("event_id"),
        F.trim(F.col("application_id")).alias("application_id"),
        F.trim(F.col("old_status")).alias("old_status"),
        F.trim(F.col("new_status")).alias("new_status"),
        F.trim(F.col("event_timestamp")).alias("event_timestamp")
    )
    .dropDuplicates(["event_id"])
)

fct_workflow_events.write.format("delta").mode("overwrite").saveAsTable(cfg["fct_workflow_events_table"])

In [0]:
spark.sql("SELECT COUNT(*) FROM fct_workflow_events").show()

spark.sql("SELECT event_id, COUNT(*) FROM fct_workflow_events GROUP BY event_id HAVING COUNT(*) > 1;").show()